# orchestration example

In [24]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama


In [25]:
### 容器，node里面的variable都必须在这里定义
class State(TypedDict):
    text: str
    category: str
    draft: str
    final: str

In [26]:
llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

# node 1 : categorize input content into 3 categories

In [27]:
def node1_classify(state: State):
    prompt = f"""
    Classify the following text into exactly one category:

    news
    email
    other

    Return only the category name.

    Text:
    {state["text"]}
    """

    response = llm.invoke(prompt)

    category = response.content.strip().lower()

    return {"category": category}

# node 2 summarize news

In [28]:
def node2_summarize_news(state: State):
    prompt = f"""
    Summarize this news article concisely.

    Focus on:
    - key event
    - important people or companies
    - business impact

    Text:
    {state["text"]}
    """

    response = llm.invoke(prompt)

    return {"draft": response.content}

# node 3 summarize emails

In [29]:
def node3_summarize_email(state: State):
    prompt = f"""
    Summarize this email.

    Identify:
    - sender's name
    - receipt's name
    - sender's main message
    - requested action
    - any deadline

    Text:
    {state["text"]}
    """

    response = llm.invoke(prompt)

    return {"draft": response.content}

# node 4 summarize other text

In [30]:
def node4_summarize_general(state: State):
    prompt = f"""
    Summarize the following text clearly and concisely.

    Text:
    {state["text"]}
    """

    response = llm.invoke(prompt)

    return {"draft": response.content}

# routing

In [31]:
## 读取第一个 LLM 的结果， 决定 LangGraph 下一步走哪个 node
def routing(state: State):
    category = state["category"]   ##input

    if "news" in category:
        return "news"

    elif "email" in category:
        return "email"

    else:
        return "other"

# node 5, reviewer 

In [32]:
def node5_review(state: State):
    prompt = f"""
    Improve the following summary.

    Make sure it is:
    - accurate
    - concise
    - easy to understand

    Original text:
    {state["text"]}

    Draft summary:
    {state["draft"]}

    Return only the improved summary.
    """

    response = llm.invoke(prompt)

    return {"final": response.content}

# connecting graph

In [33]:
builder = StateGraph(State)

builder.add_node("node1_classify", node1_classify)
builder.add_node("node2_summarize_news", node2_summarize_news)
builder.add_node("node3_summarize_email", node3_summarize_email)
builder.add_node("node4_summarize_general", node4_summarize_general)
builder.add_node("node5_review", node5_review)



builder.add_edge(START,"node1_classify")
builder.add_conditional_edges("node1_classify",
                              routing,
                              {
                                  "news": "node2_summarize_news",
                                  "email": "node3_summarize_email",
                                  "other": "node4_summarize_general",
                              }
                              )

builder.add_edge("node2_summarize_news","node5_review")
builder.add_edge("node3_summarize_email","node5_review")
builder.add_edge("node4_summarize_general","node5_review")
builder.add_edge("node5_review",END)

graph = builder.compile()

In [34]:
# RUN EXAMPLE
result = graph.invoke({
    "text": """
Hi siyi,

We have a new Handshake AI project we think you’d be a fit for. You’re receiving this because Handshake AI invites experts across many fields to help improve how AI models reason.

On Project Planck (Math), you’ll write advanced scientific questions with single, specific answers, designed to stump frontier AI models. You’ll work remotely on your own schedule. Training is provided, so no prior AI or technical experience is needed.
    """,
    "category": "",
    "draft": "",
    "final": ""
})

print(result)

{'text': '\nHi siyi,\n\nWe have a new Handshake AI project we think you’d be a fit for. You’re receiving this because Handshake AI invites experts across many fields to help improve how AI models reason.\n\nOn Project Planck (Math), you’ll write advanced scientific questions with single, specific answers, designed to stump frontier AI models. You’ll work remotely on your own schedule. Training is provided, so no prior AI or technical experience is needed.\n    ', 'category': 'other', 'draft': 'The text is an invitation from Handshake AI to participate in a project called Planck (Math). The project asks experts to create advanced scientific questions with specific answers to challenge AI models. The work can be done remotely on a flexible schedule, and training is provided.', 'final': "Here is the improved summary:\n\nHandshake AI invites experts to join Project Planck (Math), where you'll create advanced scientific questions with specific answers to challenge AI models. Work can be don

In [35]:
print('draft:'+ result['draft'])
print('final:'+ result['final'])

draft:The text is an invitation from Handshake AI to participate in a project called Planck (Math). The project asks experts to create advanced scientific questions with specific answers to challenge AI models. The work can be done remotely on a flexible schedule, and training is provided.
final:Here is the improved summary:

Handshake AI invites experts to join Project Planck (Math), where you'll create advanced scientific questions with specific answers to challenge AI models. Work can be done remotely on your own schedule, with provided training for those without prior AI experience.


In [36]:
## remove review step
builder = StateGraph(State)

builder.add_node("node1_classify", node1_classify)
builder.add_node("node2_summarize_news", node2_summarize_news)
builder.add_node("node3_summarize_email", node3_summarize_email)
builder.add_node("node4_summarize_general", node4_summarize_general)
builder.add_node("node5_review", node5_review)



builder.add_edge(START,"node1_classify")
builder.add_conditional_edges("node1_classify",
                              routing,
                              {
                                  "news": "node2_summarize_news",
                                  "email": "node3_summarize_email",
                                  "other": "node4_summarize_general",
                              }
                              )

builder.add_edge("node2_summarize_news",END)
builder.add_edge("node3_summarize_email",END)
builder.add_edge("node4_summarize_general",END)


graph2 = builder.compile()

In [37]:
# RUN EXAMPLE
result = graph2.invoke({
    "text": """
Hi siyi,

We have a new Handshake AI project we think you’d be a fit for. You’re receiving this because Handshake AI invites experts across many fields to help improve how AI models reason.

On Project Planck (Math), you’ll write advanced scientific questions with single, specific answers, designed to stump frontier AI models. You’ll work remotely on your own schedule. Training is provided, so no prior AI or technical experience is needed.
    """,
    "category": "",
    "draft": "",
#     "final": ""
})

print('draft:'+ result['draft'])

draft:The text is an invitation from Handshake AI to participate in a project called Planck (Math). The project asks experts to create advanced scientific questions with specific answers to challenge AI models. The work can be done remotely on a flexible schedule, and training is provided.
